## Análise de Dados — Grupo 7

Este notebook contém:

- Apresentação do Grupo 7  
- Objetivo do projeto  
- Descrição dos dados utilizados (Base TSE das UF's MG, RJ e SC) 
- Importação das bibliotecas  
- Processamento e limpeza  
- Análises exploratórias  
- Visualizações  
- Conclusões iniciais  

---
##  Integrantes do Grupo 7
- Cariane Ribeiro
- Leonardo Antonio
- Leonardo Rodrigues
  

---

##  Objetivo do Projeto
O objetivo deste trabalho é analisar os dados de votação, identificar padrões, comportamentos e possíveis insights relevantes para o estudo.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [ ]:
df_rj = pd.read_csv("../dados/detalhe_votacao_secao_2022_RJ.csv", encoding="latin1", sep=";")
df_mg = pd.read_csv("../dados/detalhe_votacao_secao_2022_MG.csv", encoding="latin1", sep=";")
df_sc = pd.read_csv("../dados/detalhe_votacao_secao_2022_SC.csv", encoding="latin1", sep=";")

df_rj.head(), df_mg.head(), df_sc.head()


In [ ]:
df_rj["estado"] = "RJ"
df_mg["estado"] = "MG"
df_sc["estado"] = "SC"


In [ ]:
df = pd.concat([df_rj, df_mg, df_sc], ignore_index=True)
df.head()


In [ ]:
df.info()

## Análise do Rio de Janeiro

In [ ]:
df_rj = df[df["estado"] == "RJ"].copy()


# Colunas relevantes

In [ ]:
colunas = [
    "NM_MUNICIPIO",
    "NR_ZONA",
    "NR_SECAO",
    "QT_APTOS",
    "QT_COMPARECIMENTO",
    "QT_ABSTENCOES",
    "QT_VOTOS_BRANCOS",
    "QT_VOTOS_NULOS",
    "QT_VOTOS_NOMINAIS",
    "QT_VOTOS_LEGENDA",
    "NR_TURNO",
    "CD_CARGO"
]

df_rj = df_rj[colunas].copy()



# Taxa de abstenção - Votos brancos e nulos - Votos válidos

In [ ]:
df_rj["taxa_abstencao"] = df_rj["QT_ABSTENCOES"] / df_rj["QT_APTOS"]
df_rj["prop_brancos"] = df_rj["QT_VOTOS_BRANCOS"] / df_rj["QT_APTOS"]
df_rj["prop_nulos"] = df_rj["QT_VOTOS_NULOS"] / df_rj["QT_APTOS"]
df_rj["QT_VOTOS_VALIDOS"] = df_rj["QT_VOTOS_NOMINAIS"] + df_rj["QT_VOTOS_LEGENDA"]
df_rj["prop_validos"] = df_rj["QT_VOTOS_VALIDOS"] / df_rj["QT_APTOS"]


# Como a taxa de abstenção varia entre municípios do RJ?

In [ ]:
secoes_por_municipio_rj = (
    df_rj.groupby("NM_MUNICIPIO")["NR_SECAO"]
         .nunique()
         .sort_values(ascending=False)
)

top5_secoes = secoes_por_municipio_rj.head(5)
bottom5_secoes = secoes_por_municipio_rj.tail(5)


In [ ]:
print(" 5 municípios com MAIS seções no RJ:\n")
print(top5_secoes)
print("\n 5 municípios com MENOS seções no RJ:\n")
print(bottom5_secoes)



In [ ]:
municipios_top5 = ["RIO DE JANEIRO", "NOVA IGUAÇU", "SÃO GONÇALO", "DUQUE DE CAXIAS", "MAGÉ"]
municipios_bottom5 = ["RIO DAS FLORES", "SÃO JOSÉ DE UBÁ", "LAJE DO MURIAÉ", "VARRE-SAI", "CARDOSO MOREIRA"]

municipios_10 = municipios_top5 + municipios_bottom5


In [ ]:
df_rj_10 = df_rj[df_rj["NM_MUNICIPIO"].isin(municipios_10)].copy()


In [ ]:
df_rj_10["taxa_abstencao"] = df_rj_10["QT_ABSTENCOES"] / df_rj_10["QT_APTOS"]
df_rj_10["QT_VOTOS_VALIDOS"] = df_rj_10["QT_VOTOS_NOMINAIS"] + df_rj_10["QT_VOTOS_LEGENDA"]
df_rj_10["prop_brancos"] = df_rj_10["QT_VOTOS_BRANCOS"] / df_rj_10["QT_APTOS"]
df_rj_10["prop_nulos"] = df_rj_10["QT_VOTOS_NULOS"] / df_rj_10["QT_APTOS"]
df_rj_10["prop_validos"] = df_rj_10["QT_VOTOS_VALIDOS"] / df_rj_10["QT_APTOS"]


In [ ]:
plt.figure(figsize=(12,8))
sns.scatterplot(
    data=df_rj_10,
    x="QT_APTOS",
    y="taxa_abstencao",
    hue="NM_MUNICIPIO",
    palette="tab10",
    s=80
)
plt.title("Relação entre tamanho da seção (QT_APTOS) e taxa de abstenção — 10 municípios extremos do RJ")
plt.xlabel("Tamanho da seção (QT_APTOS)")
plt.ylabel("Taxa de abstenção")
plt.legend(title="Município", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()


In [ ]:
plt.figure(figsize=(12,8))
sns.regplot(
    data=df_rj_10,
    x="QT_APTOS",
    y="taxa_abstencao",
    scatter_kws={"s": 60, "alpha": 0.6},
    line_kws={"color": "red"}
)
plt.title("Tendência entre tamanho da seção e taxa de abstenção — 10 municípios extremos do RJ")
plt.xlabel("Tamanho da seção (QT_APTOS)")
plt.ylabel("Taxa de abstenção")
plt.show()


In [ ]:
correlacao = df_rj_10["QT_APTOS"].corr(df_rj_10["taxa_abstencao"])
correlacao


In [ ]:
abst_media_10 = (
    df_rj_10.groupby("NM_MUNICIPIO")["taxa_abstencao"]
            .mean()
            .sort_values(ascending=False)
)
abst_media_10


Correlação entre tamanho da seção e abstenção:  
→ 0.032 (praticamente zero)

Abstenção média dos 10 municípios extremos:  
→ variando de 0.1898 a 0.2540

Uma correlação de 0.03 é:

extremamente baixa

praticamente nula

indica nenhuma relação linear entre tamanho da seção e abstenção
Seções maiores não têm mais abstenção.
Seções menores também não têm menos abstenção.
O tamanho da seção não explica o comportamento de abstenção.

Não existe padrão consistente que diferencie grandes e pequenos municípios.

O maior município (Rio de Janeiro) tem abstenção alta.

O menor município (Rio das Flores) também tem abstenção alta.

O menor de todos (São José de Ubá) tem a menor abstenção.

Cardoso Moreira (pequeno) tem a maior abstenção.

Ou seja:

O tamanho do município não determina a abstenção.
O tamanho da seção também não determina a abstenção.

A abstenção não é explicada por variáveis estruturais simples como tamanho da seção ou número de seções.

# Quais municípios apresentam maior proporção de votos brancos e nulos?

In [ ]:
brancos_media_10 = (
    df_rj_10.groupby("NM_MUNICIPIO")["prop_brancos"]
            .mean()
            .sort_values(ascending=False)
)

brancos_media_10


Os maiores municípios (Caxias, Nova Iguaçu, São Gonçalo, Rio) têm proporções mais altas de votos brancos.

Municípios pequenos tendem a ter proporções menores — com exceção de Rio das Flores e Laje do Muriaé, que ficam no meio da tabela.

Cardoso Moreira, apesar de ter a maior abstenção, tem a menor proporção de votos brancos.

Isso já mostra que brancos e abstenção não caminham juntos.

In [ ]:
nulos_media_10 = (
    df_rj_10.groupby("NM_MUNICIPIO")["prop_nulos"]
            .mean()
            .sort_values(ascending=False)
)

nulos_media_10


Varre-Sai, um município pequeno, lidera os nulos — comportamento bem diferente dos brancos.

Os grandes municípios novamente aparecem com valores altos.

Cardoso Moreira, de novo, aparece com valores baixos — apesar da abstenção alta.

Isso reforça que nulos também não acompanham abstenção.

In [ ]:
bn_media_10 = (
    df_rj_10.groupby("NM_MUNICIPIO")[["prop_brancos", "prop_nulos"]]
            .mean()
            .sum(axis=1)
            .sort_values(ascending=False)
)

bn_media_10


Os quatro maiores municípios do RJ (Caxias, Nova Iguaçu, São Gonçalo, Rio) lideram o ranking de brancos+nulos.

Varre-Sai, mesmo sendo pequeno, aparece com valores comparáveis aos grandes — comportamento atípico.

Cardoso Moreira, que tinha a maior abstenção, tem o menor desengajamento dentro da urna.

São José de Ubá também aparece com engajamento interno alto.

In [ ]:
plt.figure(figsize=(12,8))
bn_media_10.plot(kind="barh", color="purple")
plt.title("Proporção média de votos brancos + nulos — 10 municípios extremos do RJ")
plt.xlabel("Proporção")
plt.ylabel("Município")
plt.show()


Os municípios com maior proporção de votos brancos e nulos são, principalmente, os grandes centros urbanos: Duque de Caxias, Nova Iguaçu, São Gonçalo e Rio de Janeiro.

Municípios pequenos tendem a ter menor proporção de brancos e nulos — com exceção de Varre-Sai, que se comporta como um outlier.

Cardoso Moreira é o caso mais interessante: alta abstenção, mas baixíssimo desengajamento dentro da urna.

Isso mostra que abstenção e brancos/nulos são fenômenos diferentes, com causas distintas.

# --- Preparação dos dados  ---


In [ ]:
# Junta tudo em um único DataFrame para facilitar a análise
df_comp = pd.DataFrame({
    "abstencao": abst_media_10,
    "brancos": brancos_media_10,
    "nulos": nulos_media_10,
    "brancos_nulos": bn_media_10
})

# Ordena cada métrica
rank_abst = df_comp["abstencao"].sort_values(ascending=False)
rank_bn = df_comp["brancos_nulos"].sort_values(ascending=False)

# Identifica padrões importantes
maior_abst = rank_abst.index[0]
menor_abst = rank_abst.index[-1]
maior_bn = rank_bn.index[0]
menor_bn = rank_bn.index[-1]

# Municípios que aparecem no topo de ambos
top_abst = set(rank_abst.head(5).index)
top_bn = set(rank_bn.head(5).index)
intersecao_top = top_abst.intersection(top_bn)

# Municípios que divergem fortemente
divergentes = []
for m in df_comp.index:
    pos_abst = rank_abst.index.get_loc(m)
    pos_bn = rank_bn.index.get_loc(m)
    if abs(pos_abst - pos_bn) >= 5:  # diferença grande entre posições
        divergentes.append(m)

# --- Geração automática da conclusão ---
print("\n==============================")
print("Existe relação entre abstenção e votos brancos/nulos?")
print("==============================\n")

print("Não existe relação forte entre abstenção e votos brancos/nulos.\n"
      "Os rankings mostram que municípios com alta abstenção podem ter poucos votos brancos/nulos,\n"
      "e municípios com baixa abstenção podem ter muitos votos brancos/nulos.\n")

print(f"- Maior abstenção: {maior_abst}")
print(f"- Menor abstenção: {menor_abst}")
print(f"- Maior proporção de brancos+nulos: {maior_bn}")
print(f"- Menor proporção de brancos+nulos: {menor_bn}\n")

print("Municípios que aparecem no topo de ambos (abstenção e brancos+nulos):")
print(f"{list(intersecao_top)}\n")

print("Municípios com comportamento divergente (alta abstenção mas poucos brancos/nulos, ou vice-versa):")
print(f"{divergentes}\n")

# --- Leitura profunda ---
print("==============================")
print("LEITURA PROFUNDA")
print("==============================\n")

print(
    f"{maior_abst} tem a maior abstenção, mas aparece entre os menores níveis de brancos+nulos.\n"
    "Isso indica que quem não vota nesse município simplesmente não comparece — mas quem comparece, vota.\n\n"
    
    f"{maior_bn} lidera brancos+nulos, mas não lidera abstenção. Isso mostra um padrão oposto:\n"
    "as pessoas vão votar, mas não escolhem nenhum candidato.\n\n"
    
    "Os grandes municípios (Duque de Caxias, Nova Iguaçu, São Gonçalo e Rio de Janeiro) tendem a ter\n"
    "brancos+nulos mais altos, independentemente da abstenção. Isso sugere desengajamento dentro da urna,\n"
    "possivelmente associado a fatores urbanos, socioeconômicos e logísticos.\n\n"
    
    "Municípios pequenos têm comportamentos variados: alguns muito engajados (como São José de Ubá),\n"
    "outros com padrões semelhantes aos grandes (como Varre-Sai).\n\n"
    
    "Em resumo: abstenção e brancos/nulos são fenômenos independentes.\n"
    "Cada município tem sua própria dinâmica de engajamento eleitoral."
)


## Há diferenças relevantes entre zonas eleitorais dentro de um mesmo município?

In [ ]:
df_rj["prop_brancos_nulos"] = df_rj["prop_brancos"] + df_rj["prop_nulos"]


In [ ]:
municipios_brancos_nulos = (
    df_rj.groupby("NM_MUNICIPIO")["prop_brancos_nulos"]
    .mean()
    .sort_values(ascending=False)
)


In [ ]:
top5 = municipios_brancos_nulos.head(5).index.tolist()
bottom5 = municipios_brancos_nulos.tail(5).index.tolist()

top5, bottom5


In [ ]:
df_extremos = df_rj[df_rj["NM_MUNICIPIO"].isin(top5 + bottom5)]
df_extremos.shape


In [ ]:
zonas = (
    df_extremos.groupby(["NM_MUNICIPIO", "NR_ZONA"])
    .agg(
        taxa_abstencao=("taxa_abstencao", "mean"),
        prop_brancos_nulos=("prop_brancos_nulos", "mean"),
        votos_validos=("QT_VOTOS_VALIDOS", "sum")
    )
    .reset_index()
)

zonas.head()


In [ ]:
variacao_zonas = (
    zonas.groupby("NM_MUNICIPIO")[["taxa_abstencao", "prop_brancos_nulos"]]
    .agg(["min", "max", "mean"])
)

variacao_zonas


In [ ]:
g = sns.FacetGrid(
    zonas,
    col="NM_MUNICIPIO",
    col_wrap=5,
    height=4,
    sharey=False
)

# Boxplot para mostrar distribuição geral
g.map_dataframe(
    sns.boxplot,
    x="NR_ZONA",
    y="taxa_abstencao",
    color="lightgray",
    showcaps=True,
    boxprops={'alpha':0.6},
    whiskerprops={'alpha':0.6},
    medianprops={'color':'black'}
)

# Stripplot para mostrar cada zona como ponto
g.map_dataframe(
    sns.stripplot,
    x="NR_ZONA",
    y="taxa_abstencao",
    hue="NR_ZONA",
    palette="Set2",
    dodge=False,
    size=6,
    alpha=0.8,
    legend=False
)

g.set_titles("{col_name}")
g.set_axis_labels("Zona Eleitoral", "Taxa de Abstenção")
plt.tight_layout()
plt.show()


In [ ]:
# QUANTAS ZONAS ELEITORAIS EXISTEM POR MUNICÍPIO?

zonas_por_municipio = zonas.groupby("NM_MUNICIPIO")["NR_ZONA"].nunique()

plt.figure(figsize=(10,6))
sns.barplot(
    x=zonas_por_municipio.index,
    y=zonas_por_municipio.values,
    palette="Set2"
)
plt.xticks(rotation=45)
plt.ylabel("Número de Zonas Eleitorais")
plt.title("Quantidade de Zonas Eleitorais por Município")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

for municipio in zonas_por_municipio.index:
    df_mun = zonas[zonas["NM_MUNICIPIO"] == municipio].copy()

    # transforma zona em categoria para evitar progressão numérica
    df_mun["NR_ZONA"] = df_mun["NR_ZONA"].astype(str)

    plt.figure(figsize=(10, 5))

    df_mun_sorted = df_mun.sort_values("taxa_abstencao")

    # linha horizontal (lollipop stick)
    plt.hlines(
        y=df_mun_sorted["NR_ZONA"],
        xmin=min(df_mun_sorted["taxa_abstencao"]),
        xmax=df_mun_sorted["taxa_abstencao"],
        color="lightgray",
        linewidth=3
    )

    # bolinha colorida (lollipop head)
    plt.scatter(
        df_mun_sorted["taxa_abstencao"],
        df_mun_sorted["NR_ZONA"],
        s=200,
        c=sns.color_palette("Set2", len(df_mun_sorted)),
        edgecolor="black",
        linewidth=1
    )

    plt.title(f"Taxa de Abstenção por Zona — {municipio}", fontsize=14)
    plt.xlabel("Taxa de Abstenção")
    plt.ylabel("Zona Eleitoral")
    plt.tight_layout()
    plt.show()


## A distribuição de votos válidos por seção apresenta outliers?